[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/EelcoHoogendoorn/numga/blob/main/examples/electromagnetism/moving_charge/moving_charge.ipynb)

# Moving Charges

This example computes the electromagnetic fields of soft charges moving at constant velocity and of a point charge circling at constant speed. An opposite-charge pair at rest produces an electrostatic dipole field, whereas boosting the two charges near the speed of light flattens each charge's electric field across its path and induces a magnetic field circling its motion. A point charge on a circular orbit sheds outward electromagnetic waves that steepen from a smooth spiral at moderate speed into sharp forward-thrown pulses near the speed of light.

Each softened source is modelled at rest through closed-form spacetime derivative maps from an event step to the change in four-potential and in field, and boosting that map pulls the step into the rest frame and pushes the response back to the lab as `boost >> rest_gradient(boost << Antivector)`. Wedge-contracting basis vectors against dual steps via `(Vector ^ gradients(Antivector)).trace(1, 2)` takes the exterior derivative by incidence alone, producing the field from the potential gradient, zero from the field gradient, and the four-current from the vacuum excitation `vacuum(field)` while `t & four_current` reads the lab charge density.

In covariant tensor and Gibbs–Heaviside vector notation, boosting a static potential transforms its components by a $4 \times 4$ Lorentz matrix $A'_\mu(x') = \Lambda_\mu{}^\nu A_\nu(\Lambda^{-1} x')$, the antisymmetric curl $F_{\mu\nu} = \partial_\mu A_\nu - \partial_\nu A_\mu$ splits into $\mathbf E = -\nabla\phi - \partial_t\mathbf A$ and $\mathbf B = \nabla \times \mathbf A$, and Maxwell's equations pair metric-free curls $\nabla \times \mathbf E + \partial_t\mathbf B = 0$, $\nabla \cdot \mathbf B = 0$ with metric-contracted divergences $\partial_\mu(\eta^{\mu\alpha}\eta^{\nu\beta} F_{\alpha\beta}) = J^\nu$. For an accelerated point source on $\mathbf r_s(t)$, differentiating the Liénard–Wiechert potentials $(\phi, \mathbf A) = \frac{q}{4\pi(R - \mathbf R\cdot\boldsymbol\beta)}(1, \boldsymbol\beta)|_{t_r}$ across the implicit retarded time $t_r = t - |\mathbf x - \mathbf r_s(t_r)|$ yields nested triple cross products $\mathbf n \times ((\mathbf n - \boldsymbol\beta) \times \dot{\boldsymbol\beta}) / (1 - \mathbf n\cdot\boldsymbol\beta)^3$ for the radiation electric field alongside $\mathbf B = \mathbf n \times \mathbf E$.

In [ ]:
# In Colab, install numga and its examples from GitHub.
import sys

if "google.colab" in sys.modules:
    %pip install -q "numga[examples] @ https://github.com/EelcoHoogendoorn/numga/archive/HEAD.tar.gz"

In [ ]:
%matplotlib inline
from collections.abc import Iterator

import numpy as np

from numga import NumpyContext, stack
from numga.algebras import STA as ga
from examples.electromagnetism.moving_charge import render, scenarios
from examples.util.animation import save_figure, show_states

np.set_printoptions(precision=3, suppress=True)

mv = NumpyContext(ga).multivector
Scalar = ga.gatype.scalar()
Vector = ga.gatype.vector()
Bivector = ga.gatype.bivector()
Antivector = ga.gatype.antivector()
Rotor = ga.gatype.rotor()
PotentialGradient = ga.gatype((Vector, Antivector))                          # Vector+ <- Antivector-
FieldGradient = ga.gatype((Bivector, Antivector))                            # Bivector+ <- Antivector-

# Vacuum constitutive map from fields to excitations, the one material map where the spacetime metric enters.
vacuum = Bivector.dual()                                                     # [] Antibivector- <- Bivector+
# Unit future-pointing lab time step across the lab's spatial slice (`mv.t & time == 1`).
time = -mv.t.dual()                                                          # [] Antivector-

## Boosted potential and electromagnetic field

Spacetime events and the steps between them are modelled as intrinsic antivectors, while the four-potential is an extrinsic vector read by the steps that cross it. At rest, a Plummer-softened charge has a smooth potential along the lab's time slice `t` whose response to a spacetime step forms a closed-form map `PotentialGradient`. Boosting that rest-frame map by a rotor in the spacetime plane of motion pulls the step into the charge's rest frame via `boost << Antivector` and pushes the resulting potential change back to the lab via `boost >> rest_gradient(boost << Antivector)`. Summing the gradients of multiple charges and taking the incidence trace `(Vector ^ gradients(Antivector)).trace(1, 2)` yields the electromagnetic field bivector, while the Lorenz gauge makes the omitted divergence part of `(Vector * gradients(Antivector)).trace(1, 2)` vanish.

In [ ]:
# Opposite-charge pair placed along `y` and boosted in opposite senses along `x` at rest and high rapidity.
strengths = scenarios.dipole_strengths
positions = mv.y.dual() * scenarios.dipole_offsets                           # [2] Antivector-
directions = scenarios.dipole_directions
core_radius = scenarios.core_radius
rapidities = scenarios.dipole_rapidities
half_width, half_height = scenarios.half_width, scenarios.half_height
n_columns, n_rows = scenarios.n_columns, scenarios.n_rows

In [ ]:
def charge_profile(
    strengths: np.ndarray,
    core_radius: float,
    boosts: Rotor,
    positions: Antivector,
    events: Antivector,
) -> tuple[Antivector, Scalar, Scalar]:
    """Pulling `events` into each charge's rest frame gives the spatial separation (`[...] Antivector-`),
    the Coulomb weight per unit separation (`[...] Scalar+`), and that weight's radial slope (`[...] Scalar+`)."""
    # Undo the boost to read each event relative to the charge in the charge's own rest frame, then project out time.
    rest_offsets = boosts << (events - positions)                            # [...] Antivector-
    separations = rest_offsets - (mv.t & rest_offsets) * time                # [...] Antivector-
    squared_distance = core_radius**2 + (separations | separations)          # [...] Scalar+
    weight = strengths / (4 * np.pi) / (squared_distance * squared_distance.square_root())  # [...] Scalar+
    return separations, weight, -3 * weight / squared_distance


def potential_gradient(
    strengths: np.ndarray,
    core_radius: float,
    boosts: Rotor,
    positions: Antivector,
    events: Antivector,
) -> PotentialGradient:
    """Boosting the rest-frame Coulomb slope maps a spacetime step at each event to the change in
    four-potential (`[...] Vector+ <- Antivector-`)."""
    separations, weight, _ = charge_profile(strengths, core_radius, boosts, positions, events)  # [...] Antivector-, Scalar+
    # At rest the potential lies along `mv.t` and varies only with spatial distance from the charge.
    rest_gradient = weight * mv.t * (separations | Antivector)               # [...] Vector+ <- Antivector-
    return boosts >> rest_gradient(boosts << Antivector)                     # [...] Vector+ <- Antivector-

In [ ]:
def event_grid(half_width: float, half_height: float, n_columns: int, n_rows: int) -> Antivector:
    """Cell-centre spacetime events at lab time zero span `[-half_width, half_width]` by
    `[-half_height, half_height]` in the `z = 0` plane (`[n_rows, n_columns] Antivector-`)."""
    x = (np.arange(n_columns) + 0.5) / n_columns * 2 * half_width - half_width
    y = (np.arange(n_rows) + 0.5) / n_rows * 2 * half_height - half_height
    return (mv.x.dual() * x[None, :] + mv.y.dual() * y[:, None]).cast(Antivector)  # [n_rows, n_columns] Antivector-


def field(gradients: PotentialGradient) -> Bivector:
    """Wedge-contracting each basis vector with `gradients` gives the electromagnetic field (`[...] Bivector+`)."""
    return (Vector ^ gradients(Antivector)).trace(1, 2)                      # [...] Bivector+


boosts = (mv.xt * (rapidities[:, None] * directions / 2)).exp()              # [2, 2] Rotor+
events = event_grid(half_width, half_height, n_columns, n_rows)              # [n_rows, n_columns] Antivector-
potential_gradients = potential_gradient(
    strengths, core_radius, boosts[:, None, None, :], positions, events[..., None]
).sum(axis=-1)                                                               # [2, n_rows, n_columns] Vector+ <- Antivector-
fields = field(potential_gradients)                                          # [2, n_rows, n_columns] Bivector+
save_figure(render.fields_figure(events, fields), "moving_charge_fields");

Each rapidity case occupies a pair of panels, showing electric field strength `(field & time).norm()` in purple beside the out-of-plane magnetic component `field & xy.dual()` in red and blue. At rest in the left pair, the dipole field is purely electric; near the speed of light in the right pair, each charge's electric field contracts into a vertical band perpendicular to its motion while opposite magnetic lobes circle each path.

## Vacuum excitation and four-current

Applying the same incidence trace `(Vector ^ field_gradients(Antivector)).trace(1, 2)` to the field gradient vanishes identically, giving the homogeneous Maxwell equations without invoking the metric. For the inhomogeneous equations, the vacuum constitutive map `vacuum = Bivector.dual()` converts the field change into an excitation change whose exterior derivative yields the intrinsic four-current `Antivector-`, paired with the lab's time slice `t & four_current` to read the lab charge density. Because steps and currents are intrinsic while potentials and fields are extrinsic, mirroring the setup mirrors both the field and the four-current automatically.

In exterior-calculus notation on spacetime, these two relations read as $dF = 0$ and $d{\star F} = J$, where the Hodge star $\star$ carries the Lorentzian metric of the vacuum.

In [ ]:
# Zoomed window framing the two softened cores.
close_half_width, close_half_height = scenarios.close_half_width, scenarios.close_half_height

In [ ]:
def field_gradient(
    strengths: np.ndarray,
    core_radius: float,
    boosts: Rotor,
    positions: Antivector,
    events: Antivector,
) -> FieldGradient:
    """Boosting the rest-frame electric-field slope maps a spacetime step at each event to the change
    in field (`[...] Bivector+ <- Antivector-`)."""
    separations, weight, weight_slope = charge_profile(strengths, core_radius, boosts, positions, events)  # [...] Antivector-, Scalar+, Scalar+
    # At rest the field is purely electric, `(separations & time).dual()`; a step changes its radial weight and turns its direction.
    radial_term = weight_slope * (separations & time).dual() * (separations | Antivector)  # [...] Bivector+ <- Antivector-
    transverse_term = weight * (Antivector & time).dual()                    # [...] Bivector+ <- Antivector-
    return boosts >> (radial_term + transverse_term)(boosts << Antivector)   # [...] Bivector+ <- Antivector-


def four_current(gradients: FieldGradient) -> Antivector:
    """Wedge-contracting each basis vector with the vacuum excitation change gives the source four-current
    (`[...] Antivector-`)."""
    return (Vector ^ vacuum(gradients(Antivector))).trace(1, 2)              # [...] Antivector-


close_events = event_grid(close_half_width, close_half_height, n_columns, n_rows)  # [n_rows, n_columns] Antivector-
field_gradients = field_gradient(
    strengths, core_radius, boosts[:, None, None, :], positions, close_events[..., None]
).sum(axis=-1)                                                               # [2, n_rows, n_columns] Bivector+ <- Antivector-
four_currents = four_current(field_gradients)                                # [2, n_rows, n_columns] Antivector-
save_figure(render.densities_figure(close_events, four_currents), "moving_charge_densities");

Positive charge density appears in red and negative charge density in blue on a shared symmetric colour scale. At rest on the left, both Plummer cores are circular; at high rapidity on the right, Lorentz contraction compresses each core along its horizontal motion while time dilation raises its peak lab density.

## Accelerating and decelerating dipole

Sweeping the rapidity over a smooth half-cosine cycle from rest to the peak rapidity and back shows the continuous flattening of the electric field alongside the rise and fall of the magnetic lobes. Streaming all ninety frames on the CPU grid takes about twenty seconds.

In [ ]:
n_sweep_frames = scenarios.n_sweep_frames
duration_ms = scenarios.duration_ms

In [ ]:
def sweep_rapidities(max_rapidity: float, n_total_frames: int) -> np.ndarray:
    """A half-cosine rapidity cycle rises from rest to `max_rapidity` and returns to rest across `n_total_frames`."""
    return max_rapidity * (1 - np.cos(2 * np.pi * np.arange(n_total_frames) / n_total_frames)) / 2


def dipole_states(events: Antivector) -> Iterator[tuple[Antivector, Bivector, Antivector]]:
    """Yields the event grid, electromagnetic field, and four-current (`[n_rows, n_columns]` each) across
    one full acceleration and deceleration cycle of the opposite-charge pair."""
    for rapidity in sweep_rapidities(float(rapidities[-1]), n_sweep_frames):
        frame_boosts = (mv.xt * (rapidity * directions / 2)).exp()           # [2] Rotor+
        frame_potentials = potential_gradient(
            strengths, core_radius, frame_boosts, positions, events[..., None]
        ).sum(axis=-1)                                                       # [n_rows, n_columns] Vector+ <- Antivector-
        frame_fields = field_gradient(
            strengths, core_radius, frame_boosts, positions, events[..., None]
        ).sum(axis=-1)                                                       # [n_rows, n_columns] Bivector+ <- Antivector-
        yield events, field(frame_potentials), four_current(frame_fields)


sweep_states = show_states(
    dipole_states(events),
    render.boost_frames,
    name="moving_charge_sweep",
    duration_ms=duration_ms,
    n_frames=n_sweep_frames,
);

Orange contours mark a fixed lab charge-density level around the two cores on the left electric-strength panel, pinching into narrow vertical slivers at mid-cycle while the red and blue magnetic lobes peak on the right.

## Radiation from a circling charge

For a point charge circling at constant speed, the potential at an observation event depends on the charge's worldline at the earlier retarded emission time whose forward light cone hits that event. Fixed-point light-cone contractions close in unconditionally at subluminal speed before Newton steps polish the root. Differentiating the retarded rest-slice potential `-(orbit_charge / (4 * pi)) * rest_slice / rest_distance` with respect to the observation event gives a closed-form `PotentialGradient` whose exterior derivative captures both the bound field carried with the charge and the outgoing radiation waves.

Resolving thin relativistic pulses across a wide window requires supersampling every pixel, so the same worldline, retarded-time solver, and exterior derivative also run per sample in numga's WGSL shader dialect, where each rank-one slope `vector * (form & Antivector)` differentiates directly to `form ^ vector`.

In the covariant potential formulation of electrodynamics, this retarded potential is the Liénard–Wiechert four-potential $A_\mu(x) = \frac{q}{4\pi}\frac{u_\mu(\tau_r)}{u_\nu(\tau_r)(x - r(\tau_r))^\nu}$ evaluated at the proper time $\tau_r$ satisfying $(x - r(\tau_r))_\alpha(x - r(\tau_r))^\alpha = 0$ with $x^0 > r^0(\tau_r)$.

In [ ]:
orbit_charge = scenarios.orbit_charge
orbit_radius = scenarios.orbit_radius
orbit_speeds = scenarios.orbit_speeds
orbit_half_width = scenarios.orbit_half_width
n_orbit_pixels = scenarios.n_orbit_pixels
n_settle_steps = 6
n_newton_steps = 6
n_check_grid = 8

In [ ]:
def worldline(orbit_radius: float, orbit_speed: float, times: Scalar) -> tuple[Antivector, Antivector, Antivector]:
    """A charge circling at `orbit_speed` on `orbit_radius` traces worldline positions, lab-time
    tangents, and lab-time bends (`[...] Antivector-` each)."""
    angular_rate = orbit_speed / orbit_radius
    # Turn the starting radius in the `xy` plane, then differentiate by taking commutators with the turning plane.
    turn = (mv.xy * (-0.5 * angular_rate) * times).exp()                     # [...] Rotor+
    offset = turn >> (mv.x.dual() * orbit_radius)                            # [...] Antivector-
    tangent = time + angular_rate * offset.commutator(mv.xy)                 # [...] Antivector-
    bend = angular_rate * (tangent - time).commutator(mv.xy)                 # [...] Antivector-
    return times * time + offset, tangent, bend


def light_cone_gap(
    orbit_radius: float,
    orbit_speed: float,
    events: Antivector,
    times: Scalar,
) -> tuple[Scalar, Scalar, Scalar]:
    """Light emitted from the orbit at `times` has a time shortfall before reaching `events`, a
    shortfall rate with respect to emission time, and a spatial reach (`[...] Scalar+` each)."""
    position, tangent, _ = worldline(orbit_radius, orbit_speed, times)       # [...] Antivector-, Antivector-, Antivector-
    ray = events - position                                                  # [...] Antivector-
    spatial_ray = ray - (mv.t & ray) * time                                  # [...] Antivector-
    distance = (spatial_ray | spatial_ray).square_root()                     # [...] Scalar+
    return (mv.t & events) - times - distance, 1 - (spatial_ray | tangent) / distance, distance

In [ ]:
def retarded_time(orbit_radius: float, orbit_speed: float, events: Antivector) -> Scalar:
    """Fixed-point light-cone steps followed by Newton steps find the lab emission time (`[...] Scalar+`)
    whose forward light cone hits each event."""
    times = mv.t & events                                                    # [...] Scalar+
    for _ in range(n_settle_steps):
        _, _, distance = light_cone_gap(orbit_radius, orbit_speed, events, times)
        times = (mv.t & events) - distance                                   # [...] Scalar+
    for _ in range(n_newton_steps):
        shortfall, shortfall_rate, _ = light_cone_gap(orbit_radius, orbit_speed, events, times)
        times = times + shortfall / shortfall_rate                           # [...] Scalar+
    return times


def orbit_potential_gradient(
    orbit_charge: float,
    orbit_radius: float,
    orbit_speed: float,
    events: Antivector,
) -> PotentialGradient:
    """Differentiating the retarded potential of a circling charge with respect to the observation event
    gives the potential gradient (`[...] Vector+ <- Antivector-`)."""
    emission_times = retarded_time(orbit_radius, orbit_speed, events)        # [...] Scalar+
    position, tangent, bend = worldline(orbit_radius, orbit_speed, emission_times)  # [...] Antivector-, Antivector-, Antivector-
    # Scale lab-time derivatives by the Lorentz factor to obtain proper four-velocity and proper four-acceleration.
    dilation = 1 / np.sqrt(1 - orbit_speed**2)
    velocity, acceleration = dilation * tangent, dilation**2 * bend          # [...] Antivector-, Antivector-
    ray = events - position                                                  # [...] Antivector-
    # The charge's instantaneous spatial slice `-velocity.dual()` pairs with `velocity` to 1 and turns with `-acceleration.dual()`.
    rest_slice, slice_rate = -velocity.dual(), -acceleration.dual()          # [...] Vector+, Vector+
    rest_distance = rest_slice & ray                                         # [...] Scalar+
    # Stepping the event shifts the retarded emission time along the null ray tangent `-ray.dual()`.
    emission_slope = (-ray.dual() & Antivector) / rest_distance              # [...] Scalar+ <- Antivector-
    distance_slope = (rest_slice & Antivector) + ((slice_rate & ray) - 1) * emission_slope  # [...] Scalar+ <- Antivector-
    weight = orbit_charge / (4 * np.pi) / rest_distance                      # [...] Scalar+
    return -weight * (slice_rate * emission_slope - rest_slice * distance_slope / rest_distance)  # [...] Vector+ <- Antivector-


fast_speed = float(orbit_speeds[-1])
orbit_events = event_grid(orbit_half_width, orbit_half_width, n_check_grid, n_check_grid) + time * 3.0  # [n_check_grid, n_check_grid] Antivector-
orbit_emission_times = retarded_time(orbit_radius, fast_speed, orbit_events)  # [n_check_grid, n_check_grid] Scalar+
orbit_positions, _, _ = worldline(orbit_radius, fast_speed, orbit_emission_times)  # [n_check_grid, n_check_grid] Antivector-
orbit_potentials = orbit_potential_gradient(orbit_charge, orbit_radius, fast_speed, orbit_events)  # [n_check_grid, n_check_grid] Vector+ <- Antivector-
orbit_fields = field(orbit_potentials)                                       # [n_check_grid, n_check_grid] Bivector+

In [ ]:
orbit_field_source = f'''
const time = -mv.t.dual();
const n_settle_steps = {n_settle_steps}u;
const n_newton_steps = {n_newton_steps}u;
const pi = {np.pi!r};
var<uniform> orbit_charge: f32;
var<uniform> orbit_radius: f32;
var<uniform> orbit_speed: f32;

struct Worldline {{
    position: Antivector,
    tangent: Antivector,
    bend: Antivector,
}}

fn worldline(lab_time: f32) -> Worldline {{
    let angular_rate = orbit_speed / orbit_radius;
    let turn = (mv.xy * (-0.5 * angular_rate * lab_time)).exp();
    let offset = turn >> (mv.x.dual() * orbit_radius);
    let tangent = time + angular_rate * offset.commutator(mv.xy);
    let bend = angular_rate * (tangent - time).commutator(mv.xy);
    return Worldline(lab_time * time + offset, tangent, bend);
}}

struct LightConeGap {{
    shortfall: f32,
    rate: f32,
    reach: f32,
}}

fn light_cone_gap(event: Antivector, lab_time: f32) -> LightConeGap {{
    let source = worldline(lab_time);
    let ray = event - source.position;
    let spatial_ray = ray - (mv.t & ray) * time;
    let reach = (spatial_ray | spatial_ray).square_root();
    return LightConeGap((mv.t & event) - lab_time - reach, 1 - (spatial_ray | source.tangent) / reach, reach);
}}

fn retarded_time(event: Antivector) -> f32 {{
    var emission_time = mv.t & event;
    for (var index = 0u; index < n_settle_steps; index++) {{
        emission_time = (mv.t & event) - light_cone_gap(event, emission_time).reach;
    }}
    for (var index = 0u; index < n_newton_steps; index++) {{
        let gap = light_cone_gap(event, emission_time);
        emission_time = emission_time + gap.shortfall / gap.rate;
    }}
    return emission_time;
}}

fn orbit_field(event: Antivector) -> Bivector {{
    let source = worldline(retarded_time(event));
    let dilation = 1 / sqrt(1 - orbit_speed * orbit_speed);
    let velocity = dilation * source.tangent;
    let acceleration = dilation * dilation * source.bend;
    let ray = event - source.position;
    let rest_slice = -velocity.dual();
    let slice_rate = -acceleration.dual();
    let rest_distance = rest_slice & ray;
    let emission_form = -ray.dual() / rest_distance;
    let distance_form = rest_slice + ((slice_rate & ray) - 1) * emission_form;
    let weight = orbit_charge / (4 * pi) / rest_distance;
    return -weight * ((emission_form ^ slice_rate) - (distance_form ^ rest_slice) / rest_distance);
}}
'''

start_position = orbit_radius * mv.x.dual()                                  # [] Antivector-
save_figure(
    render.radiation_figure(
        orbit_field_source, orbit_charge, orbit_radius, orbit_speeds, start_position, orbit_half_width, n_orbit_pixels
    ),
    "moving_charge_radiation",
);

Both panels plot the out-of-plane magnetic field multiplied by radial distance from the orbit centre so outgoing waves maintain uniform amplitude across the window, with a yellow dot marking the charge at lab time zero. At half the speed of light on the left, the radiation winds outward in a smooth spiral; at nine-tenths of the speed of light on the right, relativistic beaming compresses the spiral into sharp forward-thrown arcs.

In [ ]:
# checks
spacetime_axes = stack((mv.x, mv.y, mv.z, mv.t))                             # [4] Vector+

# Homogeneous Maxwell identity and Lorenz gauge hold on the boosted dipole gradients.
homogeneous_residual = (Vector ^ field_gradients(Antivector)).trace(1, 2)    # [2, n_rows, n_columns] Antivector+
assert ((spacetime_axes ^ homogeneous_residual[..., None]).dual().abs() < 1e-11).all()
dipole_lorenz = (Vector ^ potential_gradients(Antivector).dual()).trace(1, 2)  # [2, n_rows, n_columns] Antiscalar-
assert (dipole_lorenz.dual().abs() < 1e-11).all()

# Inhomogeneous Maxwell equation matches the summed Plummer rest densities convected along each charge's boosted time step.
rest_offsets = boosts[:, None, None, :] << (close_events[..., None] - positions)  # [2, n_rows, n_columns, 2] Antivector-
separations = rest_offsets - (mv.t & rest_offsets) * time                    # [2, n_rows, n_columns, 2] Antivector-
squared_distance = core_radius**2 + (separations | separations)              # [2, n_rows, n_columns, 2] Scalar+
plummer_density = (
    3 * strengths * core_radius**2 / (4 * np.pi) / (squared_distance * squared_distance * squared_distance.square_root())
)                                                                            # [2, n_rows, n_columns, 2] Scalar+
plummer_currents = (plummer_density * (boosts[:, None, None, :] >> time)).sum(axis=-1)  # [2, n_rows, n_columns] Antivector-
assert ((spacetime_axes & (four_currents - plummer_currents)[..., None]).abs() < 1e-9).all()

# Streamed sweep states vanish magnetically at rest and amplify peak electric strength at mid-cycle.
_, rest_sweep_field, _ = sweep_states[0]
_, peak_sweep_field, _ = sweep_states[n_sweep_frames // 2]
assert ((rest_sweep_field & mv.xy.dual()).abs() < 1e-12).all()
assert ((peak_sweep_field & time).norm().max() > (rest_sweep_field & time).norm().max()).all()

# Retarded emission events lie on the past light cone and satisfy the Lorenz gauge for the circling charge.
orbit_rays = orbit_events - orbit_positions                                  # [n_check_grid, n_check_grid] Antivector-
assert ((orbit_rays | orbit_rays).abs() < 1e-10).all()
orbit_lorenz = (Vector ^ orbit_potentials(Antivector).dual()).trace(1, 2)    # [n_check_grid, n_check_grid] Antiscalar-
assert (orbit_lorenz.dual().abs() < 1e-11).all()
assert ((orbit_fields & mv.xy.dual()).abs().max() > 0).all()